# Building Transformer Language Model
## Hardware Parameters:
- GPU: NVIDIA GeForce RTX 4060 Ti 16 GB VRAM
- CPU: AMD Ryzen 7 2700X
- RAM: DDR4 16 GB

## Model Parameters:
- 

## 0. Check Compatibility of Device

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" # OK for library to be duplicated and run so

import torch

print("=== Hardware Check ===")
print(f"PyTorch Version: {torch.__version__}") # Check Version of PyTorch
print(f"CUDA Available:  {torch.cuda.is_available()}") # Check if my GPU has CUDE Available

if torch.cuda.is_available():
    # Grab Stats
    device_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    
    # Print Stats
    print(f"GPU Detected:    {device_name}")
    print(f"Total VRAM:      {vram_gb:.2f} GB")
    
    # Test tensor allocation on GPU
    x = torch.randn( 
        1000, # Rows
        1000, # Cols
        device="cuda" # Where to allocate
        )

    print("Success: Allocated memory on GPU")
else:
    print("Error: No GPU")

=== Hardware Check ===
PyTorch Version: 2.6.0+cu124
CUDA Available:  True
GPU Detected:    NVIDIA GeForce RTX 4060 Ti
Total VRAM:      16.00 GB
Success: Allocated memory on GPU


## 1. Data Preparation and Tokenization

In [2]:
import tiktoken
from datasets import load_dataset

In [3]:
# Initialize Tokenizer
gpt2_tokenizer = tiktoken.get_encoding("gpt2")
print("Vocabulary Size: ", gpt2_tokenizer.n_vocab, " tokens.")

Vocabulary Size:  50257  tokens.


In [4]:
from datasets import load_dataset, Dataset

# Stream items directly without downloading the whole repository
raw_stream = load_dataset(
    "roneneldan/TinyStories", 
    split="train", 
    streaming=True
    )

# Pull the first 5,000 items into a local list
stories_list = []
for item in raw_stream.take(5000):
    stories_list.append(item)

# Convert to standard Hugging Face Dataset format
dataset = Dataset.from_list(stories_list)
print(f"Loaded {len(dataset)} stories")

Loaded 5000 stories


In [ ]:
all_tokens = []
for story in dataset['text']:
    # Encode and append end-of-text token.
    all_tokens.extend(gpt2_tokenizer.encode(story + "<|endoftext|>", allowed_special="all"))

print(f"Total Tokens: {len(all_tokens):,}")

Total Tokens: 1,033,087


In [9]:
from torch.utils.data import Dataset as TorchDataset, DataLoader

class TextDataset(TorchDataset):
    def __init__(self, tokens, block_size):
        self.block_size = block_size
        self.num_chunks = len(tokens) // block_size
        self.tokens = tokens[:self.num_chunks * block_size + 1]

    def __len__(self):
        return self.num_chunks

    def __getitem__(self, idx):
        start = idx * self.block_size
        end = start + self.block_size

        x = torch.tensor(self.tokens[start : end], dtype=torch.long)
        y = torch.tensor(self.tokens[start + 1 : end + 1], dtype=torch.long)

        return x, y

In [10]:
# Context window: How many tokens the model can look back at to predict the next one
block_size = 128 
train_data = TextDataset(all_tokens, block_size)

# Create the DataLoader
batch_size = 16
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

In [12]:
# Test the DataLoader
xb, yb = next(iter(train_loader))
print(f"Input shape (X): {xb.shape}")
print(f"Target shape (Y): {yb.shape}")

Input shape (X): torch.Size([16, 128])
Target shape (Y): torch.Size([16, 128])


## 3. Build Transformer Architecture